# 🎓 ADAMAS UNIVERSITY
### Department of Computer Science and Engineering (AI & ML)
## Final Project: Fine-Tuning & Comparative Evaluation of Open-Source Language Models for Indian Emigration Question Answering

**Project Members:** Veerpratap Singh Tomar (UG/02/BTCSEAIML/2023/035) & Piyush Pratap Singh (UG/02/BTCSEAIML/2023/011)  
**Project Coordinator:** Dr. Tamal Ghosh  
**Academic Year:** 4th Year B.Tech. in Computer Science and Engineering (AI & ML)

---
### 📌 Complete Automated Workflow
This all-in-one notebook automates the entire lifecycle of your project:
1. **Hardware Check**: Validates available System RAM and accelerator status.
2. **Drive Mount**: Connects permanent Google Drive storage (`/content/drive/MyDrive/Research Project/`).
3. **Model Registry**: Central configuration for all 6 models (Mistral-7B, Granite-3.3-8B, Qwen3-14B + 3 upcoming).
4. **Toolchain Setup**: Installs compatible libraries, `llama.cpp` quantization tools, and `llama-server` engine.
5. **Batch Quantization**: Automatically merges LoRA adapters, converts to FP16, and quantizes to `Q4_K_M` GGUF (skips already converted models).
6. **Local Staging**: Optionally stages GGUF models on `/content` local SSD for maximum inference speed.
7. **Live Streaming UI**: Interactive Gradio Web Demo powered by `llama-server` with real-time token streaming and model selection.



In [ ]:
# 1. Environment & RAM Sanity Check
import psutil, shutil, subprocess

ram_gb = psutil.virtual_memory().total / 1e9
avail_gb = psutil.virtual_memory().available / 1e9
disk = shutil.disk_usage('/content')

print(f"🧠 System RAM  : {ram_gb:.1f} GB total ({avail_gb:.1f} GB available)")
print(f"💾 Disk Space  : {disk.free/1e9:.1f} GB free of {disk.total/1e9:.1f} GB total")

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f"⚡ Accelerator : {gpu or 'CPU / TPU Runtime active'}")

if ram_gb > 25:
    print("\n✅ High-RAM mode active (~47 GB)! 7B, 8B, and 14B models will merge and quantize safely.")
else:
    print("\nℹ️ Standard RAM active (12.7 GB). Ideal for fast GGUF inference and Gradio demo.")



In [ ]:
# 2. Mount Google Drive Storage
from google.colab import drive
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✅ Google Drive successfully mounted at /content/drive")
else:
    print("✅ Google Drive is already mounted.")



In [ ]:
# 3. Central Model Configuration & Status Audit
import os

QUANT = 'Q4_K_M'
BASE_DRIVE_DIR = '/content/drive/MyDrive/Research Project/models_for_drive'
OUT_DIR = '/content/drive/MyDrive/Research Project/gguf'
os.makedirs(OUT_DIR, exist_ok=True)

# Central registry for all 6 project models
MODELS = [
    {
        'name': 'Mistral-7B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/mistral7b',
        'base_model': 'mistralai/Mistral-7B-Instruct-v0.3',
        'enabled': True
    },
    {
        'name': 'Granite-3.3-8B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/granite33_8b',
        'base_model': 'ibm-granite/granite-3.3-8b-instruct',
        'enabled': True
    },
    {
        'name': 'Qwen3-14B-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/qwen3_14b',
        'base_model': 'Qwen/Qwen3-14B',
        'enabled': True
    },
    # --- Remaining 3 models (Arriving on Monday): ---
    {
        'name': 'Model-4-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/model4',
        'base_model': '...',
        'enabled': False
    },
    {
        'name': 'Model-5-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/model5',
        'base_model': '...',
        'enabled': False
    },
    {
        'name': 'Model-6-Finetuned',
        'adapter_dir': f'{BASE_DRIVE_DIR}/model6',
        'base_model': '...',
        'enabled': False
    },
]

print(f"🎯 Target Quantization : {QUANT}")
print(f"📁 Output GGUF Directory: {OUT_DIR}\n")
print("Model Status Audit:")
for m in MODELS:
    if m['enabled']:
        final_gguf = f"{OUT_DIR}/{m['name']}-{QUANT}.gguf"
        is_converted = os.path.exists(final_gguf)
        is_uploaded = os.path.isdir(m['adapter_dir'])
        if is_converted:
            status = f"✅ Ready ({os.path.getsize(final_gguf)/1e9:.2f} GB GGUF on Drive)"
        elif is_uploaded:
            status = "⏳ Queued (Adapter found on Drive, ready for Cell 5)"
        else:
            status = "❌ Adapter folder not found on Drive"
        print(f"  • {m['name']:<25} : {status}")



In [ ]:
%%bash
# 4. Engine & Toolchain Setup (~30s)
set -e

# A. Clean package dependencies
pip uninstall -y torchao 2>/dev/null || true
pip install -q "numpy<2.3" -U scipy

# B. Core inference & transformation dependencies
pip install -q safetensors sentencepiece protobuf peft accelerate transformers gradio requests

# C. Clone llama.cpp repository for convert_hf_to_gguf.py
if [ ! -d /content/llama.cpp ]; then
    git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
fi
pip install -q -e /content/llama.cpp/gguf-py

# D. Setup high-performance precompiled llama.cpp binaries
mkdir -p /content/llama-bin
if [ ! -x /content/llama-bin/llama-server ]; then
    echo "Downloading llama.cpp precompiled binaries..."
    wget -q https://github.com/ggml-org/llama.cpp/releases/download/b12020/llama-b12020-bin-ubuntu-x64.tar.gz -O /content/llama-bin.tar.gz
    tar -xzf /content/llama-bin.tar.gz -C /content/llama-bin --strip-components=1
    chmod +x /content/llama-bin/llama-* 2>/dev/null || true
    cp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null || true
    ldconfig
fi

export LD_LIBRARY_PATH=/content/llama-bin:${LD_LIBRARY_PATH:-}
python3 -c 'import torch, transformers, peft, gguf, gradio; print("\n🚀 All toolchains, libraries, and binaries are 100% ready!")'



In [ ]:
# 5. Automated LoRA Merge & Batch GGUF Quantization (Streamed Shard Engine - Zero OOM)
import os, gc, time, shutil, subprocess, json, torch
from safetensors.torch import load_file, save_file
from huggingface_hub import snapshot_download

# Register llama libraries with system dynamic linker to prevent libllama-common.so errors
subprocess.run('cp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null; ldconfig', shell=True)

def merge_lora_sharded(base_repo, adapter_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    print(f'   -> Locating base model snapshot for {base_repo}...')
    base_dir = snapshot_download(base_repo)

    with open(os.path.join(adapter_dir, 'adapter_config.json')) as f:
        cfg = json.load(f)
    r = cfg.get('r', 64)
    alpha = cfg.get('lora_alpha', 128)
    scale = float(alpha) / float(r)
    print(f'   -> LoRA configuration: rank={r}, alpha={alpha}, scaling factor={scale:.2f}')

    adapter_path = os.path.join(adapter_dir, 'adapter_model.safetensors')
    adapter_weights = load_file(adapter_path, device='cpu')
    print(f'   -> Loaded {len(adapter_weights)} adapter tensors ({os.path.getsize(adapter_path)/1e6:.1f} MB)')

    index_file = os.path.join(base_dir, 'model.safetensors.index.json')
    if os.path.exists(index_file):
        with open(index_file) as f:
            idx = json.load(f)
        shard_names = sorted(list(set(idx['weight_map'].values())))
    else:
        shard_names = sorted([f for f in os.listdir(base_dir) if f.endswith('.safetensors')])

    print(f'   -> Merging {len(shard_names)} weight shard(s) one-by-one (low-memory streaming)...')
    for s_idx, s_name in enumerate(shard_names, 1):
        s_path = os.path.join(base_dir, s_name)
        out_path = os.path.join(output_dir, s_name)
        print(f'      [{s_idx}/{len(shard_names)}] Shard: {s_name} ({os.path.getsize(s_path)/1e9:.2f} GB)')

        shard_tensors = load_file(s_path, device='cpu')
        merged_shard = {}
        applied = 0

        for k, v in shard_tensors.items():
            prefix = k[:-len('.weight')] if k.endswith('.weight') else k
            lora_a_key = f'base_model.model.{prefix}.lora_A.weight'
            lora_b_key = f'base_model.model.{prefix}.lora_B.weight'
            if lora_a_key not in adapter_weights:
                lora_a_key = f'{prefix}.lora_A.weight'
                lora_b_key = f'{prefix}.lora_B.weight'

            if lora_a_key in adapter_weights and lora_b_key in adapter_weights:
                a = adapter_weights[lora_a_key].float()
                b = adapter_weights[lora_b_key].float()
                delta = (b @ a) * scale
                merged_shard[k] = (v.float() + delta).to(v.dtype)
                applied += 1
            else:
                merged_shard[k] = v

        print(f'         Fused {applied} adapted layers.')
        save_file(merged_shard, out_path, metadata={'format': 'pt'})
        del shard_tensors, merged_shard
        gc.collect()

    del adapter_weights
    gc.collect()

    print('   -> Copying configs and tokenizer definitions...')
    for item in os.listdir(base_dir):
        if not item.endswith('.safetensors') and not item.endswith('.bin') and not item.startswith('.'):
            s = os.path.join(base_dir, item)
            d = os.path.join(output_dir, item)
            if os.path.isfile(s) and not os.path.exists(d):
                shutil.copy2(s, d)

    for item in os.listdir(adapter_dir):
        if (item.endswith('.json') or item.endswith('.jinja') or item.endswith('.model')) and item != 'adapter_config.json':
            shutil.copy2(os.path.join(adapter_dir, item), os.path.join(output_dir, item))

converted_models = {}

for idx, m in enumerate(MODELS, 1):
    if not m.get('enabled', False):
        continue

    m_name = m['name']
    adapter_dir = m['adapter_dir']
    base_repo = m['base_model']
    final_gguf = f"{OUT_DIR}/{m_name}-{QUANT}.gguf"

    print(f'
{"="*65}')
    print(f'[{idx}/{len(MODELS)}] Model: {m_name}')
    print(f'{"="*65}')

    if os.path.exists(final_gguf) and os.path.getsize(final_gguf) > 1e8:
        print(f'✅ Already converted: {final_gguf} ({os.path.getsize(final_gguf)/1e9:.2f} GB)')
        converted_models[m_name] = final_gguf
        continue

    if not os.path.isdir(adapter_dir):
        print(f'❌ Skipped: Adapter folder not found at {adapter_dir}')
        continue

    t_start = time.time()
    merged_dir = f'/content/{m_name}-merged'
    intermediate_gguf = f'/content/{m_name}-f16.gguf'
    staging_gguf = f'/content/{m_name}-{QUANT}.gguf'

    # Step A: Shard-by-shard streamed merge (~5 GB peak RAM, 0 MB VRAM)
    print(f'1. Merging {base_repo} with LoRA adapter...')
    merge_lora_sharded(base_repo, adapter_dir, merged_dir)

    # Step B: Convert HF merged weights to intermediate GGUF (f16)
    print('2. Converting merged weights to intermediate GGUF (f16)...')
    cmd_convert = [
        'python3', '/content/llama.cpp/convert_hf_to_gguf.py', merged_dir,
        '--outfile', intermediate_gguf,
        '--outtype', 'f16'
    ]
    res_conv = subprocess.run(cmd_convert, capture_output=True, text=True)
    if res_conv.returncode != 0:
        print('❌ Conversion to intermediate GGUF failed:', res_conv.stderr[-500:])
        shutil.rmtree(merged_dir, ignore_errors=True)
        continue

    # Free merged directory space
    shutil.rmtree(merged_dir, ignore_errors=True)

    # Step C: Quantize to Q4_K_M
    print(f'3. Quantizing to {QUANT} via llama-quantize...')
    cmd_quant = ['/content/llama-bin/llama-quantize', intermediate_gguf, staging_gguf, QUANT]
    res_quant = subprocess.run(cmd_quant, capture_output=True, text=True)
    if res_quant.returncode != 0:
        print('❌ Quantization failed:', res_quant.stderr[-500:])
        if os.path.exists(intermediate_gguf): os.remove(intermediate_gguf)
        continue

    if os.path.exists(intermediate_gguf):
        os.remove(intermediate_gguf)

    # Step D: Save final GGUF to Google Drive
    print(f'4. Saving {QUANT} GGUF to Google Drive: {final_gguf}...')
    shutil.copy2(staging_gguf, final_gguf)
    if os.path.exists(staging_gguf):
        os.remove(staging_gguf)

    elapsed = (time.time() - t_start) / 60
    size_gb = os.path.getsize(final_gguf) / 1e9
    print(f'✅ Finished in {elapsed:.1f} min! File size: {size_gb:.2f} GB')
    converted_models[m_name] = final_gguf

print(f'
{"="*65}')
print(f'Pipeline Complete! {len(converted_models)} model(s) ready in Google Drive:')
for name, path in converted_models.items():
    print(f'  • {name:<25}: {os.path.getsize(path)/1e9:.2f} GB')



In [ ]:
# 6. Optional: Stage Models on Local /content SSD for Max Speed (~15s)
import glob, shutil, os

os.makedirs('/content', exist_ok=True)
drive_ggufs = glob.glob('/content/drive/MyDrive/Research Project/gguf/*.gguf')

print("Checking for GGUF models to stage in local /content SSD:")
for f in drive_ggufs:
    fname = os.path.basename(f)
    dest = f'/content/{fname}'
    if not os.path.exists(dest):
        print(f"  • Copying {fname} to local SSD (~15s)...")
        shutil.copy2(f, dest)
        print(f"    ✅ Ready on local SSD: {dest}")
    else:
        print(f"  • Already on local SSD: {fname}")



In [ ]:
# 7. 💬 Complete Live Chat Interface with Streaming (Powered by llama-server)
import os, glob, subprocess, time, json, requests, socket

# 1. Mount Google Drive if available
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Drive notice: {e}")

# 2. Ensure Gradio is installed
try:
    import gradio as gr
except ImportError:
    os.system('pip install -q gradio')
    import gradio as gr

# 3. Setup llama.cpp server binaries
SERVER = '/content/llama-bin/llama-server'
if not os.path.exists(SERVER):
    print("Setting up llama.cpp server engine (~15s)...")
    os.makedirs('/content/llama-bin', exist_ok=True)
    os.system('wget -q https://github.com/ggml-org/llama.cpp/releases/download/b12020/llama-b12020-bin-ubuntu-x64.tar.gz -O /content/llama-bin.tar.gz')
    os.system('tar -xzf /content/llama-bin.tar.gz -C /content/llama-bin --strip-components=1')
    os.system('chmod +x /content/llama-bin/llama-*')
    os.system('cp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null || true')
    os.system('ldconfig')

os.environ['LD_LIBRARY_PATH'] = '/content/llama-bin:' + os.environ.get('LD_LIBRARY_PATH', '')

# 4. Search for GGUF models in /content (fast SSD) and Google Drive
search_paths = [
    '/content',
    '/content/drive/MyDrive/Research Project/gguf',
    '/content/drive/MyDrive/Research project/gguf',
    '/content/drive/MyDrive/gguf'
]

available_ggufs = {}
for path in search_paths:
    if os.path.exists(path):
        for f in sorted(glob.glob(f'{path}/*.gguf')):
            label = os.path.basename(f).replace('.gguf', '')
            if label not in available_ggufs:
                available_ggufs[label] = f

if not available_ggufs:
    raise FileNotFoundError("❌ No .gguf files found in /content or Google Drive!")

print(f"\n✅ Ready models found:")
for k, v in available_ggufs.items():
    print(f"  • {k} ({os.path.getsize(v)/1e9:.2f} GB) at {v}")

# 5. Dynamically assign a guaranteed free port (avoids Colab 8080 collision)
def get_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

SERVER_PORT = get_free_port()
print(f"📡 Dedicated llama-server port: {SERVER_PORT}")

# 6. Background llama-server manager (Keeps model in memory & streams responses)
current_server_proc = None
current_loaded_model = None

def ensure_server(model_label):
    global current_server_proc, current_loaded_model
    gguf_path = available_ggufs[model_label]

    if current_loaded_model == model_label and current_server_proc and current_server_proc.poll() is None:
        try:
            r = requests.get(f'http://127.0.0.1:{SERVER_PORT}/health', timeout=1)
            if r.status_code == 200:
                return True, ""
        except Exception:
            pass

    if current_server_proc:
        try:
            current_server_proc.terminate()
            current_server_proc.wait(timeout=3)
        except Exception:
            try: current_server_proc.kill()
            except Exception: pass
        current_server_proc = None

    os.system(f'fuser -k {SERVER_PORT}/tcp 2>/dev/null || true')
    time.sleep(1)

    print(f"⚡ Starting llama-server for '{model_label}' on port {SERVER_PORT}...")
    log_file = open('/tmp/llama_server.log', 'w')

    cmd = [
        SERVER,
        '-m', gguf_path,
        '--host', '127.0.0.1',
        '--port', str(SERVER_PORT),
        '-c', '2048',       # 2k context window (lightweight on RAM & CPU)
        '-t', '2'           # 2 threads matching Colab vCPU
    ]
    current_server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=log_file)
    current_loaded_model = model_label

    for i in range(45):
        if current_server_proc.poll() is not None:
            log_file.close()
            with open('/tmp/llama_server.log', 'r') as f:
                err_log = f.read()
            print(f"❌ Server exited prematurely:\n{err_log}")
            return False, err_log
        try:
            r = requests.get(f'http://127.0.0.1:{SERVER_PORT}/health', timeout=1)
            if r.status_code == 200:
                print("✅ Model loaded into RAM and llama-server is ready!")
                return True, ""
        except Exception:
            pass
        time.sleep(1)

    return False, "Server took longer than 45s to load weights."

# Pre-load the first model
model_options = list(available_ggufs.keys())
print(f"\nInitializing model '{model_options[0]}' into memory...")
ok, err = ensure_server(model_options[0])
if not ok:
    print(f"⚠️ Warning during initial pre-load: {err}")

# 7. Real-time Streaming Chat Handler (matches exact fine-tuning distribution)
def chat_handler(user_msg, history, selected_model, max_tokens, temp, top_p):
    ready, err_msg = ensure_server(selected_model)
    if not ready:
        yield f"Error loading '{selected_model}':\n{err_msg[-400:] if err_msg else 'Timed out waiting for server.'}"
        return

    # No generic system prompt (matches exact fine-tuning prompt distribution)
    messages = []
    for turn in history:
        if isinstance(turn, (list, tuple)):
            u, b = turn[0], turn[1]
            if u: messages.append({"role": "user", "content": u})
            if b: messages.append({"role": "assistant", "content": b})
        elif isinstance(turn, dict):
            messages.append({"role": turn.get("role", "user"), "content": turn.get("content", "")})
    messages.append({"role": "user", "content": user_msg})

    payload = {
        "messages": messages,
        "max_tokens": int(max_tokens),
        "temperature": float(temp),
        "top_p": float(top_p),
        "stream": True
    }

    try:
        response = requests.post(f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions", json=payload, stream=True, timeout=120)
        accumulated = ""
        for line in response.iter_lines():
            if line:
                line_str = line.decode('utf-8')
                if line_str.startswith('data: '):
                    data_str = line_str[6:].strip()
                    if data_str == '[DONE]':
                        break
                    try:
                        data = json.loads(data_str)
                        delta = data['choices'][0]['delta'].get('content', '')
                        if delta:
                            accumulated += delta
                            yield accumulated
                    except Exception:
                        pass
        if not accumulated:
            yield "No response generated by model."
    except Exception as e:
        yield f"Inference error: {e}"

# 8. Gradio UI
with gr.Blocks(title="Model Evaluation & Demo", theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    gr.Markdown("# 🚀 Fine-Tuned LLM Live Demonstration")
    gr.Markdown("Real-time streaming inference powered by `llama.cpp` server.")

    with gr.Row():
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(choices=model_options, value=model_options[0], label="🤖 Select Model", interactive=True)
            max_tokens_slider = gr.Slider(32, 512, value=192, step=16, label="Max Tokens")
            temp_slider = gr.Slider(0.0, 1.5, value=0.3, step=0.05, label="Temperature (0.2 - 0.3 for factual answers)")
            top_p_slider = gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="Top-p")
            gr.Markdown(
                "### 💡 Demo Tips\n"
                "- **Temperature 0.2 - 0.3**: Best for accurate, factual emigration answers.\n"
                "- **Ask complete questions**: e.g., *'What documents are required for an Indian student visa application?'*\n"
                "- **Streaming Active**: Instant token streaming!"
            )
        with gr.Column(scale=3):
            chat = gr.ChatInterface(
                fn=chat_handler,
                additional_inputs=[model_selector, max_tokens_slider, temp_slider, top_p_slider],
                chatbot=gr.Chatbot(height=520)
            )

demo.launch(share=True, debug=False)

